# TD3 — Mini-RAG: retrieval-augmented generation over the catalog

**~1h15 · the third brick of the PIM assistant you build across TD1–TD5.**

A supplier sends us a **new product** as raw specs — a name, a brand, a category, a price, a few attributes. There is **no Fnac-style catalog entry yet**: no short tagline, no on-brand long description. Writing one by hand for every incoming product is the chore your agent will automate in TD5. The problem: a base LLM has **never seen *our* catalog** and doesn't know *our* house style, so left alone it writes generic copy and invents specs.

**RAG (Retrieval-Augmented Generation)** fixes this. We index the **existing catalog**; for a new product we **retrieve** the most similar existing products and hand them to the model as examples, so it writes an entry that is consistent with the ones we already have. You'll build the whole pipeline end to end:

1. **embed → store** the catalog in **ChromaDB** (a real vector DB, replacing TD1's hand-rolled search);
2. **retrieve** the top-k most similar products for a query;
3. **generate** a catalog entry **without** retrieval (the baseline) and **with** retrieval (RAG);
4. **measure** whether retrieval actually helped — with a **golden dataset** + an **LLM-as-a-judge**;
5. reuse the same retriever for **catalog Q&A** — the kernel of your **mini-project**.

**How to work:** run the notebook top to bottom; complete the **6 `# TODO`** cells (**index the corpus** · `retrieve` · a **freshness demo** · `generate_entry` · `generate_entry_rag` · `judge_pairwise`); **write your own analysis answers** (you may use Claude Code to help you *think*, but the words must be yours); ask Claude Code first whenever you're stuck. Each `# TODO` ends with a quick **self-check** (an `assert` / smoke-test) so you can confirm your code is right **without any solution**.

> ⚠️ Sections 3–7 make **real Haiku API calls** — kept low on purpose: **~50–60 total** (~11 held-out products × 2 generations [no-RAG + RAG] + 1 pairwise judge each, plus a few smoke-tests). You need `ANTHROPIC_API_KEY` in a `.env` file at the project root. **Embeddings run locally** (`sentence-transformers`, free). Section 8 is **markdown-only** — zero API calls.
>
> Because the LLM is **nondeterministic**, your generated entries and the judge's verdicts will differ a little from a neighbour's or from a re-run — that's expected, and itself part of the lesson (§6). 🚀

## 1. Setup

In [1]:
import json
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import chromadb
from dotenv import load_dotenv
import anthropic

%matplotlib inline

# Shared PIM catalog — the same dataset used in TD1, TD2, TD4.
df = pd.read_csv("../data/products.csv")

# The text we index for each product: its name plus the long description (same as TD1/TD2).
df["doc"] = df["name"] + " — " + df["long_description"]

print(f"Loaded {len(df)} products across {df['category'].nunique()} categories.")

Loaded 198 products across 22 categories.


In [2]:
# Corpus vs held-out 'new arrivals' (mirrors TD2's train/test idea).
# We hold out 11 products as brand-new arrivals: their raw specs are the INPUT, and their real
# descriptions are the hidden GOLDEN reference. The rest are the indexed corpus.
# A fixed seed keeps this reproducible; the held-out set spans many categories on purpose.
# (We can't stratify across all 22 leaves with only 11 held out, so we sample at random with a seed.)
heldout_df = df.sample(n=11, random_state=0).reset_index(drop=True)
corpus_df = df.drop(df.sample(n=11, random_state=0).index).reset_index(drop=True)

print(f"Indexed corpus: {len(corpus_df)} products | Held-out new arrivals: {len(heldout_df)}")
print("Held-out categories:", sorted(heldout_df["category"].unique()))

# IMPORTANT: the held-out products are NOT in the corpus, so retrieve() can never return a product's
# own entry -> no self-retrieval, no cheating.
assert set(heldout_df["sku"]).isdisjoint(set(corpus_df["sku"])), "held-out leaked into the corpus"

Indexed corpus: 187 products | Held-out new arrivals: 11
Held-out categories: ['Bluetooth Speakers', 'Books', 'Headphones', 'Smartphones', 'Smartwatches', 'Televisions', 'Vacuum Cleaners', 'Video Games', 'Vinyl Records', 'Wireless Earbuds']


In [3]:
# LLM client (Haiku only) + local embedding model (free, from TD1).
load_dotenv()  # reads ANTHROPIC_API_KEY from .env at the project root
if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY not found. Create a .env file at the project root with\n"
        "ANTHROPIC_API_KEY=sk-ant-...  (see resources/setup_guide.md). "
        "Embeddings are local and work without a key, but sections 3-7 need one."
    )
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Client + embedding model ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Client + embedding model ready.


In [4]:
# Helper: a new arrival's 'specs' = the raw fields a supplier would send.
# We deliberately WITHHOLD short_description and long_description -- those are what the model must write.
def specs_of(row):
    """Return the raw supplier specs for a product row (no descriptions)."""
    return {
        "name": row["name"],
        "brand": row["brand"],
        "category": row["category"],
        "price": float(row["price"]),
        "attributes": json.loads(row["attributes"]),
    }

# Peek at two held-out 'new arrivals' as the supplier would hand them to us.
for _, r in heldout_df.head(2).iterrows():
    print(json.dumps(specs_of(r), indent=2))
    print("(hidden golden short_description:", repr(r["short_description"]), ")\n")

{
  "name": "JBL Charge 5",
  "brand": "JBL",
  "category": "Bluetooth Speakers",
  "price": 159.0,
  "attributes": {
    "color": "Black",
    "power_output_w": 40,
    "battery_life_hours": 20,
    "water_resistance": "IP67",
    "weight_g": 960
  }
}
(hidden golden short_description: 'Rugged portable speaker that doubles as a power bank.' )

{
  "name": "Gone Girl",
  "brand": "Orion Publishing",
  "category": "Books",
  "price": 10.99,
  "attributes": {
    "author": "Gillian Flynn",
    "genre": "Thriller",
    "format": "Paperback",
    "pages": 463,
    "language": "English"
  }
}
(hidden golden short_description: 'A twisting thriller about a marriage gone wrong.' )



## 2. From a hand-rolled search to a vector DB

In **TD1** you wrote `top_k_similar` by hand: embed everything, cosine-score a query against every row, sort. That's perfect for a demo, but it doesn't **persist** (you re-embed on every run) and it doesn't **scale** (brute-force over millions of vectors). A **vector database** solves both — it stores embeddings, indexes them for fast nearest-neighbour search, and survives restarts.

We'll use **ChromaDB**. Note one key choice: we compute embeddings **ourselves** with the same local MiniLM model as TD1 and hand the vectors to Chroma explicitly — we do **not** let Chroma pick its own embedder. That keeps the **one shared vector space** running through TD1 → TD3 → TD4 → TD5.

**Your first TODO** is exactly this — embed the corpus and load it into Chroma. The boilerplate (client + collection + the metadata to attach) is given; you write the embed-and-`add` step, then a self-check confirms every record landed with its embedding, document and metadata.

In [5]:
# Build an in-memory Chroma collection and INDEX the corpus -- this is the RAG retriever's storage.
# In-memory keeps the notebook reproducible top-to-bottom (a fresh index every run). Your mini-project
# will swap this for a PERSISTENT index (see the brief in section 8).
chroma_client = chromadb.Client()
# Drop any stale collection so re-running this cell is clean. (given)
if "catalog" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("catalog")
collection = chroma_client.create_collection("catalog")

# The metadata we attach to each product -- ALL the corpus_df fields, so retrieve() hands back the
# WHOLE product (sku is the Chroma id; doc is the indexed text). (given)
metadatas = [{"name": r["name"], "brand": r["brand"], "category": r["category"],
              "price": float(r["price"]),
              "short_description": r["short_description"],
              "long_description": r["long_description"],
              "attributes": r["attributes"]}   # JSON string -> Chroma metadata must be scalar
             for _, r in corpus_df.iterrows()]

# Embed the corpus docs ourselves with MiniLM, then hand Chroma the vectors explicitly.
corpus_embeddings = embed_model.encode(corpus_df["doc"].tolist()).tolist()
collection.add(
    ids=corpus_df["sku"].tolist(),
    embeddings=corpus_embeddings,
    documents=corpus_df["doc"].tolist(),
    metadatas=metadatas,
)

# Self-check once implemented: the collection must hold every corpus product, each carrying an
# embedding, a document AND its full metadata -- exactly what retrieve() will read back later.
assert collection.count() == len(corpus_df), "every corpus product should be indexed"
stored = collection.get(include=["embeddings", "documents", "metadatas"])
assert len(stored["embeddings"]) == len(corpus_df), "one embedding per product"
assert len(stored["embeddings"][0]) == 384, "should be MiniLM's 384-dim vectors (not Chroma's own embedder)"
assert all(len(d) > 0 for d in stored["documents"]), "every record should carry its document text"
required = {"name", "brand", "category", "price", "short_description", "long_description", "attributes"}
assert all(required <= set(m) for m in stored["metadatas"]), "every record needs the full metadata"
print(f"Indexed {collection.count()} products into ChromaDB — embeddings, documents & metadata all present.")

Indexed 187 products into ChromaDB — embeddings, documents & metadata all present.


In [6]:
def retrieve(query_text, k=3):
    """Return the k corpus products most similar to `query_text`, as a list of dicts.
    Steps:
      1. embed `query_text` with `embed_model` (MiniLM) -- same model used to index the corpus;
      2. query the Chroma `collection` with that embedding for the k nearest neighbours
         (collection.query(query_embeddings=[vec], n_results=k));
      3. return a list of k dicts -- one per hit -- each carrying the product's FULL stored
         metadata: the sku (the Chroma id) PLUS every field kept in that hit's metadata
         (here: name, brand, category, price, short_description, long_description, attributes).
         Don't hand-pick a
         couple of fields -- return all of them, so a caller gets the whole product.
    The conceptual point: you embed the QUERY with the SAME model as the corpus, so query and
    documents share one vector space and nearest-neighbour = most semantically similar.
    """
    query_embedding = embed_model.encode(query_text).tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=k)
    hits = []
    for i in range(len(results["ids"][0])):
        hit = {"sku": results["ids"][0][i]}
        hit.update(results["metadatas"][0][i])
        hits.append(hit)
    return hits

# Self-check once implemented: querying a corpus product's OWN text must return that product first.
probe = corpus_df.iloc[0]
hits = retrieve(probe["doc"], k=3)
assert len(hits) == 3
assert hits[0]["sku"] == probe["sku"], "the nearest neighbour of a doc should be itself"
assert set(hits[0]) >= {"sku", "name", "brand", "category", "price", "short_description", "long_description", "attributes"}
print("retrieve self-check passed. Top hit:", hits[0]["name"])

retrieve self-check passed. Top hit: Sony WH-1000XM5


In [7]:
# FRESHNESS DEMO (the RAG aha, shown not told): add ONE brand-new made-up product, then
# retrieve a matching query a moment later -- it should come back immediately, with NO retraining.
fresh_doc = "AquaBeat Pro — waterproof floating bluetooth pool speaker with 20-hour battery and RGB lights."
collection.add(   # (given) index the new product, exactly like you indexed the corpus above
    ids=["SKU-NEW-001"],
    embeddings=[embed_model.encode(fresh_doc).tolist()],
    documents=[fresh_doc],
    metadatas=[{"name": "AquaBeat Pro", "brand": "AquaBeat", "category": "Bluetooth Speakers",
                "price": 79.0,
                "short_description": "Floating waterproof party speaker.",
                "long_description": fresh_doc,
                "attributes": "{}"}],   # same metadata schema as the corpus above
)

# We just indexed 'AquaBeat Pro' a moment ago, with NO retraining. Prove it's already findable:
# retrieve the top-3 for a query that never names the product.
hits = retrieve("a speaker I can use in the swimming pool", k=3)

# Self-check once implemented: the freshly indexed product must be retrievable straight away.
assert any(h["sku"] == "SKU-NEW-001" for h in hits), "the new product should be findable the instant it's indexed"
print("After adding the new product, query 'speaker I can use in the swimming pool':")
for h in hits:
    print(f"  - {h['name']}  ({h['category']})")

# Remove it again so the rest of the notebook works on the clean 187-product corpus.
collection.delete(ids=["SKU-NEW-001"])
print("\n(Removed the demo product; corpus back to", collection.count(), "items.)")

After adding the new product, query 'speaker I can use in the swimming pool':
  - AquaBeat Pro  (Bluetooth Speakers)
  - Bose SoundLink Flex  (Bluetooth Speakers)
  - Tribit StormBox Micro 2  (Bluetooth Speakers)

(Removed the demo product; corpus back to 187 items.)


> ✅ **Takeaway.** You now have a **persistent semantic index** — the **RAG retriever**. It replaces TD1's numpy loop with a real vector DB, and the freshness demo showed the headline property of retrieval: a brand-new product becomes findable **the instant you index it — no retraining**. In **TD4** you'll wrap exactly this `retrieve` function as the MCP tool **`search_products`**.

## 3. Generate an entry WITHOUT retrieval — the baseline

Now the generation half. For a held-out **new arrival**, we hand the model **only the raw specs** and ask it to write the customer-facing entry: a short tagline, a long description, and any attributes it would suggest. No catalog context at all — this is the baseline we'll compare RAG against.

We use **structured output** (the same technique as TD2): we declare the JSON shape we want and have the SDK enforce it, so every reply is a dict with the fields `{short_description, long_description, suggested_attributes}` — no parsing, no surprises.

In [8]:
from pydantic import BaseModel
from typing import Dict

class CatalogEntry(BaseModel):
    short_description: str          # one-line tagline
    long_description: str           # 2-4 sentence catalog description
    suggested_attributes: Dict[str, str] = {}   # e.g. {'color': 'Black', 'connectivity': 'Bluetooth'}

def specs_block(specs):
    """Render a new arrival's raw specs as a readable text block for a prompt. (given)"""
    attrs = ", ".join(f'{k}: {v}' for k, v in specs['attributes'].items())
    return (
        f"Name: {specs['name']}\n"
        f"Brand: {specs['brand']}\n"
        f"Category: {specs['category']}\n"
        f"Price: EUR {specs['price']:.0f}\n"
        f"Attributes: {attrs}"
    )
print(specs_block(specs_of(heldout_df.iloc[0])))

Name: JBL Charge 5
Brand: JBL
Category: Bluetooth Speakers
Price: EUR 159
Attributes: color: Black, power_output_w: 40, battery_life_hours: 20, water_resistance: IP67, weight_g: 960


In [9]:
def generate_entry(specs):
    """Write a catalog entry from raw specs ALONE (no retrieval). Return a CatalogEntry.
    Steps:
      1. build a prompt that gives the model ONLY the raw specs (use specs_block(specs)) and asks it
         to write a customer-facing entry: a short tagline, a 2-4 sentence long description, and any
         attributes worth surfacing;
      2. call client.messages.parse(model=MODEL, max_tokens=512, messages=[...],
         output_format=CatalogEntry) -- structured output, same as TD2;
      3. return resp.parsed_output (a validated CatalogEntry).
    The hard part is the PROMPT, not the call. (Instruction only -- no solution here.)
    """
    prompt = (
        "You are writing a customer-facing catalog entry for an e-commerce website, based only on "
        "the raw supplier specs below (no other context is available). Write a short one-line "
        "tagline (short_description), a 2-4 sentence long description (long_description), and any "
        "attributes worth surfacing to a shopper (suggested_attributes). Do not invent specs that "
        "aren't given.\n\n"
        f"{specs_block(specs)}"
    )
    resp = client.messages.parse(
        model=MODEL,
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}],
        output_format=CatalogEntry,
    )
    return resp.parsed_output

# Self-check once implemented (makes 1 API call):
e0 = generate_entry(specs_of(heldout_df.iloc[0]))
assert isinstance(e0, CatalogEntry)
assert e0.short_description and e0.long_description
print("short:", e0.short_description)
print("long :", e0.long_description)

short: Powerful 40W portable Bluetooth speaker with 20-hour battery life and IP67 water resistance.
long : The JBL Charge 5 delivers 40W of audio power in a portable design weighing 960g. With up to 20 hours of battery life, it keeps the music playing all day long. Its IP67 water resistance rating makes it durable enough for outdoor adventures and wet environments.


In [10]:
# Pre-built: run the baseline on 1-2 held-out items and print the generated entry NEXT TO the real one.
def show_entry(title, entry):
    print(f"--- {title} ---")
    print("  short:", entry.short_description if isinstance(entry, CatalogEntry) else entry["short"])
    print("  long :", entry.long_description if isinstance(entry, CatalogEntry) else entry["long"])

for i in range(2):
    row = heldout_df.iloc[i]
    print(f"\n==== {row['name']}  ({row['category']}) ====")
    gen = generate_entry(specs_of(row))
    show_entry("NO-RAG (specs only)", gen)
    show_entry("REAL catalog entry",
               {"short": row["short_description"], "long": row["long_description"]})


==== JBL Charge 5  (Bluetooth Speakers) ====
--- NO-RAG (specs only) ---
  short: Powerful portable Bluetooth speaker with 20-hour battery life and waterproof design
  long : The JBL Charge 5 is a rugged Bluetooth speaker delivering 40W of sound output, perfect for on-the-go listening. With an impressive 20-hour battery life, you can enjoy extended playtime without frequent recharging. Built with IP67 water resistance, it's ready for outdoor adventures, poolside sessions, and any environment. Weighing just 960g, it strikes the perfect balance between portability and powerful audio performance.
--- REAL catalog entry ---
  short: Rugged portable speaker that doubles as a power bank.
  long : Presented by Fnac.fr. The Charge 5 delivers loud, balanced sound with an IP67 rating for outdoor use, plus a USB port to charge your phone. Up to 20 hours of playtime makes it a dependable companion for trips and gatherings.

==== Gone Girl  (Books) ====
--- NO-RAG (specs only) ---
  short: A gripp

Look at the baseline next to the real entry. The model **understands** the product, but with no catalog context the copy tends to read **generic**, drift in **tone**, and sometimes **assert specs** the supplier never gave us. Nothing told it what *our* entries sound like. That's what retrieval will fix in §4.

## 4. Generate WITH retrieval (RAG)

Same specs, same schema — but first we **retrieve the k most similar existing products** and drop their real entries into the prompt as **exemplars**. The model imitates the house style by example, instead of guessing it. This is **dynamic few-shot by retrieval**: the examples aren't hand-picked, they're whatever the retriever judges most relevant to *this* product.

The conceptual core of the TODO is the **prompt assembly**: how do you present the k exemplars so the model *imitates the voice* (length, tone, what to mention) **without copying** their specifics?

In [11]:
def generate_entry_rag(specs, k=3):
    """Write a catalog entry using RAG. Return a CatalogEntry.
    Steps:
      1. retrieve(...) the k most similar EXISTING products. Build the query text the same way the
         corpus was indexed -- name + ' — ' + a short spec summary works well (so the query lands
         near similar products in the shared vector space);
      2. format those k retrieved entries as EXEMPLARS in the prompt (show their short + long
         descriptions) and instruct the model to MATCH their style/tone/length for the NEW product --
         WITHOUT copying their specifics;
      3. then give the NEW product's raw specs (specs_block) and ask for the entry;
      4. call client.messages.parse(..., output_format=CatalogEntry) with max_tokens=512 and return
         resp.parsed_output.
    The hard part is step 2 -- the exemplar-augmented prompt. (Instruction only -- no solution here.)
    """
    query_text = specs["name"] + " — " + specs_block(specs)
    exemplars = retrieve(query_text, k=k)
    exemplars_block = "\n\n".join(
        f"Example {i+1} (existing catalog entry):\n"
        f"  short_description: {ex['short_description']}\n"
        f"  long_description: {ex['long_description']}"
        for i, ex in enumerate(exemplars)
    )
    prompt = (
        "You are writing a customer-facing catalog entry for an e-commerce website. Below are "
        "existing catalog entries for products SIMILAR to the new one. Study their style, tone, "
        "structure and length -- this is the house voice -- and MATCH it for the new product, "
        "WITHOUT copying their specific facts (those examples describe different products).\n\n"
        f"{exemplars_block}\n\n"
        f"Now write the entry for this NEW product, using ONLY its own specs below for content:\n"
        f"{specs_block(specs)}\n\n"
        "Write a short one-line tagline (short_description), a 2-4 sentence long description "
        "(long_description) matching the house style above, and any attributes worth surfacing "
        "(suggested_attributes)."
    )
    resp = client.messages.parse(
        model=MODEL,
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}],
        output_format=CatalogEntry,
    )
    return resp.parsed_output

# Self-check once implemented (makes 1 API call):
r0 = generate_entry_rag(specs_of(heldout_df.iloc[0]))
assert isinstance(r0, CatalogEntry)
assert r0.short_description and r0.long_description
print("RAG short:", r0.short_description)
print("RAG long :", r0.long_description)

RAG short: Robust portable speaker with powerful sound and all-day battery.
RAG long : Presented by Fnac.fr. The JBL Charge 5 delivers 40W of punchy audio in a rugged, IP67-rated design built for adventure. With 20 hours of battery life and a durable build that handles splashes and drops, it's equally at home by the pool, on a hiking trail, or powering a gathering indoors.


In [13]:
# Pre-built: the VISIBLE ablation -- NO-RAG vs RAG vs REAL, side by side, for the same 1-2 items.
for i in range(2):
    row = heldout_df.iloc[i]
    print(f"\n==== {row['name']}  ({row['category']}) ====")
    show_entry("NO-RAG (specs only)", generate_entry(specs_of(row)))
    show_entry("RAG  (+ retrieved exemplars)", generate_entry_rag(specs_of(row)))
    show_entry("REAL catalog entry",
               {"short": row["short_description"], "long": row["long_description"]})


==== JBL Charge 5  (Bluetooth Speakers) ====
--- NO-RAG (specs only) ---
  short: Powerful 40W Bluetooth speaker with 20-hour battery life and IP67 waterproofing
  long : The JBL Charge 5 is a robust Bluetooth speaker delivering 40W of powerful sound. With an impressive 20-hour battery life, it keeps the music playing all day long. Its IP67 water resistance rating makes it perfect for outdoor adventures, while the compact 960g design ensures easy portability.
--- RAG  (+ retrieved exemplars) ---
  short: Rugged portable speaker with powerful sound and all-day battery.
  long : Presented by Fnac.fr. The JBL Charge 5 combines 40W of powerful audio with an IP67-rated, durable design built for outdoor adventure. Its 20-hour battery keeps the music going all day, while the compact form factor makes it easy to transport. Whether poolside, on the trail, or at a picnic, it delivers robust sound without compromise.
--- REAL catalog entry ---
  short: Rugged portable speaker that doubles as a p

### The house style, made visible

Did you spot it? Every **real** catalog entry opens its `long_description` with the exact same Fnac byline — *"Presented by Fnac.fr."*. It's a house-style rule, and **nobody put it in the prompt**. The no-RAG model has never seen our catalog; the RAG model only sees the retrieved exemplars — which all happen to carry the byline.

> 🔮 **Predict (10 seconds):** which version will reproduce the byline? Then run the cell.

The RAG model will reproduce the byline

In [14]:
# The style marker test: does each generated long_description open with the Fnac house byline?
# Nobody told the model this rule -- no-RAG can't know it; RAG can only have learned it from the exemplars.
MARKER = "Presented by Fnac.fr"
print(f"{'product':32} {'NO-RAG':>8} {'RAG':>8}   <- opens with the Fnac byline?")
for i in range(3):
    row = heldout_df.iloc[i]
    specs = specs_of(row)
    no_rag_long = generate_entry(specs).long_description
    rag_long = generate_entry_rag(specs).long_description
    has_no_rag = "YES" if MARKER in no_rag_long else "no"
    has_rag = "YES" if MARKER in rag_long else "no"
    print(f"{row['name'][:32]:32} {has_no_rag:>8} {has_rag:>8}")

product                            NO-RAG      RAG   <- opens with the Fnac byline?
JBL Charge 5                           no      YES
Gone Girl                              no      YES
Apple Watch Series 9 (45mm, GPS)       no      YES


**The byline shows up only with RAG.** The base model can't invent a convention it never saw; retrieval hands it three live examples that all start *"Presented by Fnac.fr."*, and the model copies the pattern — without being told the rule exists. That's the whole point of **dynamic few-shot by retrieval**: a fixed-string byline is the most visible case, but the *same* mechanism transfers tone, length and phrasing you could never spell out in a static rule.

> ✅ **Takeaway.** Read the three columns. The retrieved entries **are** the brand voice: RAG copy lands closer to the real entries in tone and length, and sticks to the given specs — all from in-context examples, with **no separate "tone" document and no fine-tuning**. That's **dynamic few-shot by retrieval**. But eyeballing two items isn't *measurement* — §5 turns this into a number.

## 5. But is it actually better? — golden dataset + LLM-as-a-judge

Eyeballing two entries is an anecdote, not a measurement. And here's the catch flagged back in TD2: for *generated* text there is **no single correct description**, so we **cannot compute accuracy**. What we *do* have is the held-out products' **real, human-written entries** — our **golden dataset**.

To compare at scale we use an **LLM-as-a-judge**: show the judge both candidate entries (no-RAG vs RAG) plus the specs and the real entry as ground truth, and ask which is better. The hard part of the TODO is the **rubric** — what you tell the judge to optimise for.

In [ ]:
class Verdict(BaseModel):
    winner: str   # must be one of "A", "B", "tie"
    reason: str

def judge_pairwise(specs, real_entry, entry_A, entry_B):
    """Ask Haiku which candidate entry is better. Return a Verdict.
    `real_entry` is a dict {'short': ..., 'long': ...} (the golden human-written entry);
    `entry_A` and `entry_B` are CatalogEntry objects (the two candidates, slot order decided by the caller).
    Steps:
      1. write a RUBRIC prompt: show the specs (specs_block), the REAL entry as the ground-truth
         reference, and the two candidates A and B; ask which is better -- more on-brand (closer to
         the real entry's voice), more faithful to the specs, no invented facts;
      2. call client.messages.parse(model=MODEL, max_tokens=256, temperature=0, messages=[...],
         output_format=Verdict). temperature=0 makes the judge as repeatable as possible;
      3. return resp.parsed_output.
    The hard part is the RUBRIC, not the call. (Instruction only -- no solution here.)
    """
    prompt = (
        "You are a strict, impartial judge comparing two AI-generated catalog entries written for "
        "the SAME product, against a REAL human-written reference entry for that product. Judge "
        "which candidate (A or B) is better on: (1) on-brand -- how close its tone, structure and "
        "length are to the REAL reference entry; (2) faithful -- it sticks to the given specs and "
        "invents no facts. If the two are equally good, answer 'tie'.\n\n"
        f"Product specs:\n{specs_block(specs)}\n\n"
        f"REAL reference entry (ground truth, house style):\n"
        f"  short_description: {real_entry['short']}\n"
        f"  long_description: {real_entry['long']}\n\n"
        f"Candidate A:\n"
        f"  short_description: {entry_A.short_description}\n"
        f"  long_description: {entry_A.long_description}\n\n"
        f"Candidate B:\n"
        f"  short_description: {entry_B.short_description}\n"
        f"  long_description: {entry_B.long_description}\n\n"
        "Which is better: 'A', 'B', or 'tie'? Give a one-sentence reason."
    )
    resp = client.messages.parse(
        model=MODEL,
        max_tokens=256,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
        output_format=Verdict,
    )
    return resp.parsed_output

# Self-check once implemented (makes 1 API call): winner must be one of the 3 allowed values.
row = heldout_df.iloc[0]
real = {"short": row["short_description"], "long": row["long_description"]}
v = judge_pairwise(specs_of(row), real, generate_entry(specs_of(row)), generate_entry_rag(specs_of(row)))
assert v.winner in {"A", "B", "tie"}
print("winner:", v.winner, "| reason:", v.reason)

> 🔮 **Predict first (10 seconds — just for you, not graded).** Before you run the eval set below: out of the ~11 held-out products, **what fraction of the time do you expect the RAG entry to win** over the no-RAG baseline? Jot a number, then check it against the result.

In [ ]:
# Pre-built EVAL-SET RUNNER: for every held-out item, generate both ways, then judge.
# We ALTERNATE the A/B slot by item parity so the RAG entry isn't always 'A' -- this controls the
# judge's position bias (and sets up the discussion in section 6). We map the winner back to RAG/no-RAG.
records = []
for i, (_, row) in enumerate(heldout_df.iterrows()):
    specs = specs_of(row)
    real = {"short": row["short_description"], "long": row["long_description"]}
    no_rag = generate_entry(specs)
    rag = generate_entry_rag(specs)
    # Even items: RAG is A, no-RAG is B. Odd items: swap. (position-bias control)
    if i % 2 == 0:
        entry_A, entry_B, A_is_rag = rag, no_rag, True
    else:
        entry_A, entry_B, A_is_rag = no_rag, rag, False
    v = judge_pairwise(specs, real, entry_A, entry_B)
    if v.winner == "tie":
        outcome = "tie"
    else:
        picked_rag = (v.winner == "A") == A_is_rag
        outcome = "rag" if picked_rag else "no_rag"
    records.append({"name": row["name"], "category": row["category"],
                    "outcome": outcome, "reason": v.reason,
                    "no_rag_short": no_rag.short_description, "rag_short": rag.short_description})
eval_df = pd.DataFrame(records)
print(f"Judged {len(eval_df)} held-out products.")

In [ ]:
# Pre-built: tally and plot the win-rate (RAG wins / ties / no-RAG wins).
counts = eval_df["outcome"].value_counts()
n = len(eval_df)
n_rag   = int(counts.get("rag", 0))
n_tie   = int(counts.get("tie", 0))
n_norag = int(counts.get("no_rag", 0))
print(f"RAG wins : {n_rag}/{n}  ({n_rag/n:.0%})")
print(f"Ties     : {n_tie}/{n}  ({n_tie/n:.0%})")
print(f"No-RAG   : {n_norag}/{n}  ({n_norag/n:.0%})")

plt.figure(figsize=(6, 3.5))
plt.bar(["RAG wins", "Tie", "No-RAG wins"], [n_rag, n_tie, n_norag],
        color=["#2a9d8f", "#e9c46a", "#e76f51"])
plt.ylabel("# held-out products")
plt.title("LLM-as-a-judge: RAG vs no-RAG (golden-set comparison)")
plt.tight_layout()
plt.show()

> ✅ **Takeaway.** Retrieval **measurably wins** — RAG takes a clear majority of the head-to-head comparisons. And notice what you just did: you **measured the quality of a generative system that has no ground-truth labels**, by building a small golden set and letting a judge model compare against it. That technique — not the exact win-rate — is the transferable skill.

**Question.** Look at **your own** run. (a) What RAG win-rate did you get? It's probably a near-clean sweep. (b) Recall §4: every real entry — and therefore every RAG entry — opens with the *"Presented by Fnac.fr."* byline, while no-RAG never produces it. The judge sees that same byline in the REAL reference entry. So how much of RAG's win do you think comes from genuinely **better copy**, versus from simply **matching that one surface marker** the judge is comparing against? (c) Does a near-100% score make you trust the judge **more** or **less**? Name one change to the experiment that would tell you whether RAG's win is *real* or just the byline.

**Your answer** — write it yourself. You may use AI to help you think it through, but the written answer must be in your own words.

_..._

## 6. The honest caveats of LLM-as-a-judge

Before you trust that number, be honest about the instrument. The judge is **itself an LLM**, and it carries known biases:

- **Position bias** — it can favour whichever entry is shown *first* (or *last*). We mitigated this by **alternating the A/B slot by item parity** in §5 (so the RAG entry is "A" for half the items and "B" for the other half), then mapping the verdict back to RAG/no-RAG. Without that, a position-biased judge could manufacture a win-rate out of thin air.
- **Verbosity bias** — judges tend to prefer the *longer* answer, even when it's not better.
- **Self-preference** — a model tends to prefer text that *it* (or a similar model) wrote.

On top of that the judge is **nondeterministic** (even at `temperature=0`, runs can differ), it would need its **own validation against human judgement** before you trust it in production, and our **golden set is tiny** (~11 items) — a single flipped verdict swings the rate by ~9 points. This is exactly the open-ended-evaluation problem flagged in **TD2 §7**, now concrete: measuring generation is the hard part, and the measuring instrument needs measuring too.

**Question.** Your **mini-project** (§8) is a catalog **Q&A chatbot** — again a task with no single correct answer. (a) What would a **golden dataset** for it look like — what's an *input*, and what's the *reference* (given there's no one gold string)? (b) How would you adapt this **pairwise judge** to grade *answers* instead of *entries*? (c) What would you check **before trusting** the judge's verdict?

**Your answer** — write it yourself. You may use AI to help you think it through, but the written answer must be in your own words.

_..._

## 7. Bridge → your mini-project: retrieve-to-answer

The **same retriever** powers a second use: **answering questions about the catalog**. The loop is tiny — `retrieve` the relevant products, stuff them into a prompt, and let Haiku answer **grounded in them** (so it talks about *our* actual catalog, not products in general). This `retrieve → answer` loop is the kernel of your mini-project.

In [ ]:
# Pre-built (short): retrieve-to-answer. Grounded catalog Q&A in ~5 lines.
def answer_question(question, k=4):
    """Answer a question about the catalog, grounded in the k most relevant products."""
    hits = retrieve(question, k=k)
    context = "\n".join(
        f"- {h['name']} ({h['category']}): {h['short_description']}" for h in hits
    )
    prompt = (
        "Answer the question using ONLY the catalog products listed below. If nothing fits, say so.\n\n"
        f"Catalog products:\n{context}\n\nQuestion: {question}"
    )
    resp = client.messages.create(
        model=MODEL, max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text

print(answer_question("do we carry wireless earbuds under 100 euros?"))

*This `retrieve → answer` loop is the core of your mini-project — see the brief below.*

## 8. Mini-project brief — a Fnac-style catalog chatbot *(build it yourself)*

*(Markdown only — no code here. You build this **from scratch** with Claude Code; we ship no scaffold. The whole point is that you can.)*

### The goal
A small **Flask web app** — a Fnac.com-style **product chatbot**. A single page with a text box: a user types a question about the catalog (*"do you have noise-cancelling headphones under €200?"*) and gets back a grounded answer. It's the §7 `retrieve → answer` loop, behind a web UI.

### Where to build it
Create a **`mini_project/`** folder next to this notebook (`notebooks/TD3_rag/mini_project/`) and put **all** the mini-project code there — `build_index.py`, the Flask app, `requirements.txt`, its own `README.md`, and the persistent ChromaDB index it writes. Keep it self-contained in that one folder.

### Two pieces to build (name them exactly)

1. **`build_index.py`** — a script that loads `products.csv` (the shared dataset, at `../../data/products.csv` from your `mini_project/` folder), embeds each product with MiniLM (`all-MiniLM-L6-v2`), and builds a **persistent** ChromaDB index on disk. You run it **once**.
   - *Why persistent?* The notebook used an **in-memory** Chroma client so it re-runs clean — but a real web app can't re-embed the whole catalog on every request or every restart. A persistent index (`chromadb.PersistentClient(path=...)`) is built once and reloaded instantly. Add the MiniLM vectors **explicitly**, exactly as you did in §2 — keep the one shared vector space.

2. **the Flask app** — loads the persistent index at startup, exposes a page (and/or a JSON endpoint) that takes a question, runs **`retrieve → grounded-answer`** (the `answer_question` logic from §7), and renders the answer.

### Reuse what you built
`retrieve` (§2) and `answer_question` (§7) **are** the kernel — lift them straight into the app. The only real change is swapping the in-memory Chroma client for a persistent one.

### Ground rules
- **Haiku only** (`claude-haiku-4-5`) for the LLM call.
- **No API key in the code** — load it from `.env` (just like this notebook).
- Ship a **`requirements.txt`** and a short **`README.md`** with run instructions: `python build_index.py`, then run the app.
- Keep it **minimal** — a working POC, not a product.

> *Ask Claude Code to scaffold the Flask boilerplate for you; **you** own the RAG logic.*

## 9. Wrap-up

What you take away from TD3 — three transferable techniques:

- **RAG.** Index your data, retrieve the relevant pieces at query time, and inject them into the prompt. The model uses **your** knowledge without any retraining — and a freshly indexed item is usable immediately.
- **Dynamic few-shot by retrieval.** The retrieved entries *are* the brand voice. Instead of writing a static "tone guide", you let the retriever pick the most relevant live examples for each input.
- **LLM-as-a-judge.** When there's no ground truth, build a small **golden dataset** and let a judge model *measure* quality — while staying honest about the judge's own biases (position, verbosity, self-preference) and the noise of a small set.

**Where these return:** in **TD4** you wrap `retrieve` as the MCP tool **`search_products`**; in **TD5** the agent calls it to enrich incoming products end-to-end. Your mini-project is the first standalone app built on this kernel.

### 🚀 Going further (optional — no solutions; use Claude Code)
- **Tune `k`.** Re-run the eval set with `k=1`, `k=3`, `k=5` retrieved exemplars. Does more context keep helping, or does the win-rate plateau (or drop) as irrelevant neighbours creep in?
- **A pointwise judge.** Instead of pairwise A/B, score each entry **1–5** on *on-brand*, *faithful*, and *consistent* (structured output again). Does the pointwise ranking agree with the pairwise verdict? Where do they disagree, and why?
- **Probe the judge's biases.** Artificially **pad** the no-RAG entry with extra length and re-judge — does the win-rate move? That's verbosity bias you can *see*. ✅